# HW07 - Кластеризация, внутренние метрики качества, PCA/t-SNE

Домашнее задание по теме: кластеризация, внутренние метрики качества, PCA/t-SNE и "честный" unsupervised-эксперимент на синтетических данных.

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.cluster import adjusted_rand_score

import warnings
warnings.filterwarnings('ignore')

# Настройка визуализации
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Загрузка и первичный анализ данных

In [ ]:
# Загрузка данных
ds1_path = 'data/S07-hw-dataset-01.csv'
ds2_path = 'data/S07-hw-dataset-02.csv'
ds3_path = 'data/S07-hw-dataset-03.csv'

df1 = pd.read_csv(ds1_path)
df2 = pd.read_csv(ds2_path)
df3 = pd.read_csv(ds3_path)

print("Dataset 1 shape:", df1.shape)
print("Dataset 2 shape:", df2.shape)
print("Dataset 3 shape:", df3.shape)

In [ ]:
# Информация о датасете 1
print("Dataset 1 info:")
print(df1.info())
print("\nDataset 1 head:")
print(df1.head())
print("\nDataset 1 describe:")
print(df1.describe())
print("\nDataset 1 missing values:")
print(df1.isnull().sum())

In [ ]:
# Информация о датасете 2
print("Dataset 2 info:")
print(df2.info())
print("\nDataset 2 head:")
print(df2.head())
print("\nDataset 2 describe:")
print(df2.describe())
print("\nDataset 2 missing values:")
print(df2.isnull().sum())

In [ ]:
# Информация о датасете 3
print("Dataset 3 info:")
print(df3.info())
print("\nDataset 3 head:")
print(df3.head())
print("\nDataset 3 describe:")
print(df3.describe())
print("\nDataset 3 missing values:")
print(df3.isnull().sum())

## Подготовка данных для каждого датасета

In [ ]:
# Подготовка данных для датасета 1
sample_ids_1 = df1['sample_id']
X1 = df1.drop(['sample_id'], axis=1)

# Определение числовых и категориальных признаков
numeric_features_1 = X1.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_1 = X1.select_dtypes(include=['object']).columns.tolist()

print(f"Dataset 1 - Numeric features: {len(numeric_features_1)}, Categorical features: {len(categorical_features_1)}")
print(f"Numeric: {numeric_features_1}")
print(f"Categorical: {categorical_features_1}")

In [ ]:
# Подготовка данных для датасета 2
sample_ids_2 = df2['sample_id']
X2 = df2.drop(['sample_id'], axis=1)

# Определение числовых и категориальных признаков
numeric_features_2 = X2.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_2 = X2.select_dtypes(include=['object']).columns.tolist()

print(f"Dataset 2 - Numeric features: {len(numeric_features_2)}, Categorical features: {len(categorical_features_2)}")
print(f"Numeric: {numeric_features_2}")
print(f"Categorical: {categorical_features_2}")

In [ ]:
# Подготовка данных для датасета 3
sample_ids_3 = df3['sample_id']
X3 = df3.drop(['sample_id'], axis=1)

# Определение числовых и категориальных признаков
numeric_features_3 = X3.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_3 = X3.select_dtypes(include=['object']).columns.tolist()

print(f"Dataset 3 - Numeric features: {len(numeric_features_3)}, Categorical features: {len(categorical_features_3)}")
print(f"Numeric: {numeric_features_3}")
print(f"Categorical: {categorical_features_3}")

## Препроцессинг данных

In [ ]:
# Препроцессинг для датасета 1
preprocessor_1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_1),
    ],
    remainder='passthrough'
)

X1_processed = preprocessor_1.fit_transform(X1)
# Если есть категориальные признаки, преобразуем в правильный формат
if categorical_features_1:
    X1_processed = pd.DataFrame(X1_processed, columns=numeric_features_1+categorical_features_1)
else:
    X1_processed = pd.DataFrame(X1_processed, columns=numeric_features_1)

print(f"Dataset 1 processed shape: {X1_processed.shape}")

In [ ]:
# Препроцессинг для датасета 2
preprocessor_2 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_2),
    ],
    remainder='passthrough'
)

X2_processed = preprocessor_2.fit_transform(X2)
if categorical_features_2:
    X2_processed = pd.DataFrame(X2_processed, columns=numeric_features_2+categorical_features_2)
else:
    X2_processed = pd.DataFrame(X2_processed, columns=numeric_features_2)

print(f"Dataset 2 processed shape: {X2_processed.shape}")

In [ ]:
# Препроцессинг для датасета 3
preprocessor_3 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_3),
    ],
    remainder='passthrough'
)

X3_processed = preprocessor_3.fit_transform(X3)
if categorical_features_3:
    X3_processed = pd.DataFrame(X3_processed, columns=numeric_features_3+categorical_features_3)
else:
    X3_processed = pd.DataFrame(X3_processed, columns=numeric_features_3)

print(f"Dataset 3 processed shape: {X3_processed.shape}")

## Кластеризация для датасета 1

In [ ]:
# KMeans для датасета 1
range_n_clusters = list(range(2, 21))
silhouette_scores_kmeans_1 = []
db_scores_kmeans_1 = []
ch_scores_kmeans_1 = []

for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X1_processed)
    
    silhouette_avg = silhouette_score(X1_processed, cluster_labels)
    silhouette_scores_kmeans_1.append(silhouette_avg)
    
    db_score = davies_bouldin_score(X1_processed, cluster_labels)
    db_scores_kmeans_1.append(db_score)
    
    ch_score = calinski_harabasz_score(X1_processed, cluster_labels)
    ch_scores_kmeans_1.append(ch_score)

# Визуализация результатов для KMeans на датасете 1
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(range_n_clusters, silhouette_scores_kmeans_1, marker='o')
plt.title('Silhouette Score vs Number of Clusters (DS1)')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')

plt.subplot(1, 3, 2)
plt.plot(range_n_clusters, db_scores_kmeans_1, marker='o')
plt.title('Davies-Bouldin Score vs Number of Clusters (DS1)')
plt.xlabel('Number of Clusters')
plt.ylabel('Davies-Bouldin Score')

plt.subplot(1, 3, 3)
plt.plot(range_n_clusters, ch_scores_kmeans_1, marker='o')
plt.title('Calinski-Harabasz Score vs Number of Clusters (DS1)')
plt.xlabel('Number of Clusters')
plt.ylabel('Calinski-Harabasz Score')

plt.tight_layout()
plt.show()

In [ ]:
# Определение оптимального числа кластеров для датасета 1 (по silhouette score)
optimal_k_1 = range_n_clusters[np.argmax(silhouette_scores_kmeans_1)]
print(f"Optimal number of clusters for DS1 (by Silhouette): {optimal_k_1}")

# Обучение KMeans с оптимальным числом кластеров
kmeans_1_opt = KMeans(n_clusters=optimal_k_1, random_state=42, n_init=10)
labels_kmeans_1 = kmeans_1_opt.fit_predict(X1_processed)

# Метрики для оптимального решения
silhouette_kmeans_1 = silhouette_score(X1_processed, labels_kmeans_1)
db_kmeans_1 = davies_bouldin_score(X1_processed, labels_kmeans_1)
ch_kmeans_1 = calinski_harabasz_score(X1_processed, labels_kmeans_1)

print(f"KMeans DS1 - Silhouette: {silhouette_kmeans_1:.3f}, DB: {db_kmeans_1:.3f}, CH: {ch_kmeans_1:.3f}")

In [ ]:
# DBSCAN для датасета 1
eps_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
min_samples_values = [2, 3, 4, 5, 6, 7, 8]

best_silhouette_dbscan_1 = -1
best_eps_1 = None
best_min_samples_1 = None
best_labels_dbscan_1 = None

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X1_processed)
        
        # Учитываем, что DBSCAN может помечать точки как шум (-1)
        unique_labels = set(labels)
        if len(unique_labels) > 1 and -1 not in unique_labels or list(unique_labels) != [-1]:
            try:
                silhouette = silhouette_score(X1_processed, labels)
                if silhouette > best_silhouette_dbscan_1:
                    best_silhouette_dbscan_1 = silhouette
                    best_eps_1 = eps
                    best_min_samples_1 = min_samples
                    best_labels_dbscan_1 = labels
            except:
                continue

if best_labels_dbscan_1 is not None:
    db_dbscan_1 = davies_bouldin_score(X1_processed, best_labels_dbscan_1)
    ch_dbscan_1 = calinski_harabasz_score(X1_processed, best_labels_dbscan_1)
    noise_ratio_1 = (best_labels_dbscan_1 == -1).sum() / len(best_labels_dbscan_1)
    
    print(f"DBSCAN DS1 - Best params: eps={best_eps_1}, min_samples={best_min_samples_1}")
    print(f"DBSCAN DS1 - Silhouette: {best_silhouette_dbscan_1:.3f}, DB: {db_dbscan_1:.3f}, CH: {ch_dbscan_1:.3f}")
    print(f"DBSCAN DS1 - Noise ratio: {noise_ratio_1:.3f}")
else:
    print("DBSCAN did not find any clusters for DS1")

In [ ]:
# AgglomerativeClustering для датасета 1
linkages = ['ward', 'complete', 'average', 'single']
best_silhouette_agg_1 = -1
best_k_agg_1 = None
best_linkage_1 = None
best_labels_agg_1 = None

for linkage in linkages:
    for n_clusters in range_n_clusters:
        # Ward работает только с евклидовым расстоянием
        if linkage == 'ward':
            agg_cluster = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
        else:
            agg_cluster = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage, connectivity=None)
        
        labels = agg_cluster.fit_predict(X1_processed)
        
        silhouette = silhouette_score(X1_processed, labels)
        if silhouette > best_silhouette_agg_1:
            best_silhouette_agg_1 = silhouette
            best_k_agg_1 = n_clusters
            best_linkage_1 = linkage
            best_labels_agg_1 = labels

db_agg_1 = davies_bouldin_score(X1_processed, best_labels_agg_1)
ch_agg_1 = calinski_harabasz_score(X1_processed, best_labels_agg_1)

print(f"Agglomerative DS1 - Best params: n_clusters={best_k_agg_1}, linkage={best_linkage_1}")
print(f"Agglomerative DS1 - Silhouette: {best_silhouette_agg_1:.3f}, DB: {db_agg_1:.3f}, CH: {ch_agg_1:.3f}")

## Кластеризация для датасета 2

In [ ]:
# KMeans для датасета 2
range_n_clusters = list(range(2, 21))
silhouette_scores_kmeans_2 = []
db_scores_kmeans_2 = []
ch_scores_kmeans_2 = []

for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X2_processed)
    
    silhouette_avg = silhouette_score(X2_processed, cluster_labels)
    silhouette_scores_kmeans_2.append(silhouette_avg)
    
    db_score = davies_bouldin_score(X2_processed, cluster_labels)
    db_scores_kmeans_2.append(db_score)
    
    ch_score = calinski_harabasz_score(X2_processed, cluster_labels)
    ch_scores_kmeans_2.append(ch_score)

# Визуализация результатов для KMeans на датасете 2
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(range_n_clusters, silhouette_scores_kmeans_2, marker='o')
plt.title('Silhouette Score vs Number of Clusters (DS2)')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')

plt.subplot(1, 3, 2)
plt.plot(range_n_clusters, db_scores_kmeans_2, marker='o')
plt.title('Davies-Bouldin Score vs Number of Clusters (DS2)')
plt.xlabel('Number of Clusters')
plt.ylabel('Davies-Bouldin Score')

plt.subplot(1, 3, 3)
plt.plot(range_n_clusters, ch_scores_kmeans_2, marker='o')
plt.title('Calinski-Harabasz Score vs Number of Clusters (DS2)')
plt.xlabel('Number of Clusters')
plt.ylabel('Calinski-Harabasz Score')

plt.tight_layout()
plt.show()

In [ ]:
# Определение оптимального числа кластеров для датасета 2 (по silhouette score)
optimal_k_2 = range_n_clusters[np.argmax(silhouette_scores_kmeans_2)]
print(f"Optimal number of clusters for DS2 (by Silhouette): {optimal_k_2}")

# Обучение KMeans с оптимальным числом кластеров
kmeans_2_opt = KMeans(n_clusters=optimal_k_2, random_state=42, n_init=10)
labels_kmeans_2 = kmeans_2_opt.fit_predict(X2_processed)

# Метрики для оптимального решения
silhouette_kmeans_2 = silhouette_score(X2_processed, labels_kmeans_2)
db_kmeans_2 = davies_bouldin_score(X2_processed, labels_kmeans_2)
ch_kmeans_2 = calinski_harabasz_score(X2_processed, labels_kmeans_2)

print(f"KMeans DS2 - Silhouette: {silhouette_kmeans_2:.3f}, DB: {db_kmeans_2:.3f}, CH: {ch_kmeans_2:.3f}")

In [ ]:
# DBSCAN для датасета 2
eps_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
min_samples_values = [2, 3, 4, 5, 6, 7, 8]

best_silhouette_dbscan_2 = -1
best_eps_2 = None
best_min_samples_2 = None
best_labels_dbscan_2 = None

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X2_processed)
        
        unique_labels = set(labels)
        if len(unique_labels) > 1 and -1 not in unique_labels or list(unique_labels) != [-1]:
            try:
                silhouette = silhouette_score(X2_processed, labels)
                if silhouette > best_silhouette_dbscan_2:
                    best_silhouette_dbscan_2 = silhouette
                    best_eps_2 = eps
                    best_min_samples_2 = min_samples
                    best_labels_dbscan_2 = labels
            except:
                continue

if best_labels_dbscan_2 is not None:
    db_dbscan_2 = davies_bouldin_score(X2_processed, best_labels_dbscan_2)
    ch_dbscan_2 = calinski_harabasz_score(X2_processed, best_labels_dbscan_2)
    noise_ratio_2 = (best_labels_dbscan_2 == -1).sum() / len(best_labels_dbscan_2)
    
    print(f"DBSCAN DS2 - Best params: eps={best_eps_2}, min_samples={best_min_samples_2}")
    print(f"DBSCAN DS2 - Silhouette: {best_silhouette_dbscan_2:.3f}, DB: {db_dbscan_2:.3f}, CH: {ch_dbscan_2:.3f}")
    print(f"DBSCAN DS2 - Noise ratio: {noise_ratio_2:.3f}")
else:
    print("DBSCAN did not find any clusters for DS2")

In [ ]:
# AgglomerativeClustering для датасета 2
linkages = ['ward', 'complete', 'average', 'single']
best_silhouette_agg_2 = -1
best_k_agg_2 = None
best_linkage_2 = None
best_labels_agg_2 = None

for linkage in linkages:
    for n_clusters in range_n_clusters:
        if linkage == 'ward':
            agg_cluster = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
        else:
            agg_cluster = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage, connectivity=None)
        
        labels = agg_cluster.fit_predict(X2_processed)
        
        silhouette = silhouette_score(X2_processed, labels)
        if silhouette > best_silhouette_agg_2:
            best_silhouette_agg_2 = silhouette
            best_k_agg_2 = n_clusters
            best_linkage_2 = linkage
            best_labels_agg_2 = labels

db_agg_2 = davies_bouldin_score(X2_processed, best_labels_agg_2)
ch_agg_2 = calinski_harabasz_score(X2_processed, best_labels_agg_2)

print(f"Agglomerative DS2 - Best params: n_clusters={best_k_agg_2}, linkage={best_linkage_2}")
print(f"Agglomerative DS2 - Silhouette: {best_silhouette_agg_2:.3f}, DB: {db_agg_2:.3f}, CH: {ch_agg_2:.3f}")

## Кластеризация для датасета 3

In [ ]:
# KMeans для датасета 3
range_n_clusters = list(range(2, 21))
silhouette_scores_kmeans_3 = []
db_scores_kmeans_3 = []
ch_scores_kmeans_3 = []

for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X3_processed)
    
    silhouette_avg = silhouette_score(X3_processed, cluster_labels)
    silhouette_scores_kmeans_3.append(silhouette_avg)
    
    db_score = davies_bouldin_score(X3_processed, cluster_labels)
    db_scores_kmeans_3.append(db_score)
    
    ch_score = calinski_harabasz_score(X3_processed, cluster_labels)
    ch_scores_kmeans_3.append(ch_score)

# Визуализация результатов для KMeans на датасете 3
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(range_n_clusters, silhouette_scores_kmeans_3, marker='o')
plt.title('Silhouette Score vs Number of Clusters (DS3)')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')

plt.subplot(1, 3, 2)
plt.plot(range_n_clusters, db_scores_kmeans_3, marker='o')
plt.title('Davies-Bouldin Score vs Number of Clusters (DS3)')
plt.xlabel('Number of Clusters')
plt.ylabel('Davies-Bouldin Score')

plt.subplot(1, 3, 3)
plt.plot(range_n_clusters, ch_scores_kmeans_3, marker='o')
plt.title('Calinski-Harabasz Score vs Number of Clusters (DS3)')
plt.xlabel('Number of Clusters')
plt.ylabel('Calinski-Harabasz Score')

plt.tight_layout()
plt.show()

In [ ]:
# Определение оптимального числа кластеров для датасета 3 (по silhouette score)
optimal_k_3 = range_n_clusters[np.argmax(silhouette_scores_kmeans_3)]
print(f"Optimal number of clusters for DS3 (by Silhouette): {optimal_k_3}")

# Обучение KMeans с оптимальным числом кластеров
kmeans_3_opt = KMeans(n_clusters=optimal_k_3, random_state=42, n_init=10)
labels_kmeans_3 = kmeans_3_opt.fit_predict(X3_processed)

# Метрики для оптимального решения
silhouette_kmeans_3 = silhouette_score(X3_processed, labels_kmeans_3)
db_kmeans_3 = davies_bouldin_score(X3_processed, labels_kmeans_3)
ch_kmeans_3 = calinski_harabasz_score(X3_processed, labels_kmeans_3)

print(f"KMeans DS3 - Silhouette: {silhouette_kmeans_3:.3f}, DB: {db_kmeans_3:.3f}, CH: {ch_kmeans_3:.3f}")

In [ ]:
# DBSCAN для датасета 3
eps_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
min_samples_values = [2, 3, 4, 5, 6, 7, 8]

best_silhouette_dbscan_3 = -1
best_eps_3 = None
best_min_samples_3 = None
best_labels_dbscan_3 = None

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X3_processed)
        
        unique_labels = set(labels)
        if len(unique_labels) > 1 and -1 not in unique_labels or list(unique_labels) != [-1]:
            try:
                silhouette = silhouette_score(X3_processed, labels)
                if silhouette > best_silhouette_dbscan_3:
                    best_silhouette_dbscan_3 = silhouette
                    best_eps_3 = eps
                    best_min_samples_3 = min_samples
                    best_labels_dbscan_3 = labels
            except:
                continue

if best_labels_dbscan_3 is not None:
    db_dbscan_3 = davies_bouldin_score(X3_processed, best_labels_dbscan_3)
    ch_dbscan_3 = calinski_harabasz_score(X3_processed, best_labels_dbscan_3)
    noise_ratio_3 = (best_labels_dbscan_3 == -1).sum() / len(best_labels_dbscan_3)
    
    print(f"DBSCAN DS3 - Best params: eps={best_eps_3}, min_samples={best_min_samples_3}")
    print(f"DBSCAN DS3 - Silhouette: {best_silhouette_dbscan_3:.3f}, DB: {db_dbscan_3:.3f}, CH: {ch_dbscan_3:.3f}")
    print(f"DBSCAN DS3 - Noise ratio: {noise_ratio_3:.3f}")
else:
    print("DBSCAN did not find any clusters for DS3")

In [ ]:
# AgglomerativeClustering для датасета 3
linkages = ['ward', 'complete', 'average', 'single']
best_silhouette_agg_3 = -1
best_k_agg_3 = None
best_linkage_3 = None
best_labels_agg_3 = None

for linkage in linkages:
    for n_clusters in range_n_clusters:
        if linkage == 'ward':
            agg_cluster = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
        else:
            agg_cluster = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage, connectivity=None)
        
        labels = agg_cluster.fit_predict(X3_processed)
        
        silhouette = silhouette_score(X3_processed, labels)
        if silhouette > best_silhouette_agg_3:
            best_silhouette_agg_3 = silhouette
            best_k_agg_3 = n_clusters
            best_linkage_3 = linkage
            best_labels_agg_3 = labels

db_agg_3 = davies_bouldin_score(X3_processed, best_labels_agg_3)
ch_agg_3 = calinski_harabasz_score(X3_processed, best_labels_agg_3)

print(f"Agglomerative DS3 - Best params: n_clusters={best_k_agg_3}, linkage={best_linkage_3}")
print(f"Agglomerative DS3 - Silhouette: {best_silhouette_agg_3:.3f}, DB: {db_agg_3:.3f}, CH: {ch_agg_3:.3f}")

## Визуализация результатов с использованием PCA

In [ ]:
# PCA визуализация для лучшего решения каждого датасета
pca = PCA(n_components=2, random_state=42)

# PCA для датасета 1 (используем лучший метод)
X1_pca = pca.fit_transform(X1_processed)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
scatter = plt.scatter(X1_pca[:, 0], X1_pca[:, 1], c=labels_kmeans_1, cmap='viridis', alpha=0.7)
plt.title(f'Dataset 1 - KMeans (k={optimal_k_1})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter)

# PCA для датасета 2
X2_pca = pca.fit_transform(X2_processed)
plt.subplot(1, 3, 2)
scatter = plt.scatter(X2_pca[:, 0], X2_pca[:, 1], c=labels_kmeans_2, cmap='viridis', alpha=0.7)
plt.title(f'Dataset 2 - KMeans (k={optimal_k_2})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter)

# PCA для датасета 3
X3_pca = pca.fit_transform(X3_processed)
plt.subplot(1, 3, 3)
scatter = plt.scatter(X3_pca[:, 0], X3_pca[:, 1], c=labels_kmeans_3, cmap='viridis', alpha=0.7)
plt.title(f'Dataset 3 - KMeans (k={optimal_k_3})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter)

plt.tight_layout()
plt.show()

# Сохранение графиков в папку artifacts/figures
plt.savefig('artifacts/figures/pca_dataset_1.png')
plt.savefig('artifacts/figures/pca_dataset_2.png')
plt.savefig('artifacts/figures/pca_dataset_3.png')

In [ ]:
# Визуализация параметров для датасета 1
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(range_n_clusters, silhouette_scores_kmeans_1, marker='o', label='KMeans')
plt.title('Silhouette Score vs Number of Clusters (DS1)')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.axvline(x=optimal_k_1, color='red', linestyle='--', label=f'Optimal k={optimal_k_1}')
plt.legend()

plt.subplot(1, 3, 2)
if best_labels_dbscan_1 is not None:
    silhouette_scores_dbscan_1 = []
    for eps in eps_values[:4]:  # Берем только часть значений для наглядности
        dbscan = DBSCAN(eps=eps, min_samples=best_min_samples_1)
        labels = dbscan.fit_predict(X1_processed)
        unique_labels = set(labels)
        if len(unique_labels) > 1 and -1 not in unique_labels or list(unique_labels) != [-1]:
            try:
                silhouette = silhouette_score(X1_processed, labels)
                silhouette_scores_dbscan_1.append(silhouette)
            except:
                silhouette_scores_dbscan_1.append(-1)
        else:
            silhouette_scores_dbscan_1.append(-1)
    
    plt.plot(eps_values[:4], silhouette_scores_dbscan_1, marker='o', label='DBSCAN')
    plt.title('Silhouette Score vs Eps (DS1)')
    plt.xlabel('Eps')
    plt.ylabel('Silhouette Score')
    plt.axhline(y=best_silhouette_dbscan_1, color='red', linestyle='--', label=f'Best score={best_silhouette_dbscan_1:.3f}')
    plt.legend()
else:
    plt.text(0.5, 0.5, 'DBSCAN did not find clusters', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
    plt.title('DBSCAN - No clusters found')

plt.subplot(1, 3, 3)
linkage_scores = []
for linkage in linkages:
    if linkage == 'ward':
        agg_cluster = AgglomerativeClustering(n_clusters=optimal_k_1, linkage=linkage)
    else:
        agg_cluster = AgglomerativeClustering(n_clusters=optimal_k_1, linkage=linkage, connectivity=None)
    labels = agg_cluster.fit_predict(X1_processed)
    silhouette = silhouette_score(X1_processed, labels)
    linkage_scores.append(silhouette)

plt.bar(linkages, linkage_scores)
plt.title('Silhouette Score vs Linkage (DS1)')
plt.xlabel('Linkage')
plt.ylabel('Silhouette Score')
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('artifacts/figures/ds1_param_comparison.png')
plt.show()

## Проверка устойчивости (на примере датасета 1)

In [ ]:
# Проверка устойчивости KMeans на датасете 1
n_runs = 5
random_states = [42, 123, 456, 789, 999]
ari_scores = []

# Первый запуск как эталон
reference_labels = KMeans(n_clusters=optimal_k_1, random_state=random_states[0], n_init=10).fit_predict(X1_processed)

# Сравнение с другими запусками
for rs in random_states[1:]:
    current_labels = KMeans(n_clusters=optimal_k_1, random_state=rs, n_init=10).fit_predict(X1_processed)
    ari = adjusted_rand_score(reference_labels, current_labels)
    ari_scores.append(ari)

print(f"ARI scores between different runs: {ari_scores}")
print(f"Mean ARI: {np.mean(ari_scores):.3f} (+/- {np.std(ari_scores):.3f})")

# Визуализация устойчивости
plt.figure(figsize=(8, 5))
plt.bar(range(len(ari_scores)), ari_scores)
plt.title('Stability Check - ARI Scores between Different Random States (DS1)')
plt.xlabel('Run Index')
plt.ylabel('Adjusted Rand Index')
plt.axhline(y=np.mean(ari_scores), color='red', linestyle='--', label=f'Mean ARI={np.mean(ari_scores):.3f}')
plt.legend()
plt.savefig('artifacts/figures/stability_check.png')
plt.show()

## Итоги для каждого датасета

In [ ]:
# Сводка по датасету 1
print("=== DATASET 1 SUMMARY ===")
print(f"KMeans - k={optimal_k_1}: Silhouette={silhouette_kmeans_1:.3f}, DB={db_kmeans_1:.3f}, CH={ch_kmeans_1:.3f}")
if best_labels_dbscan_1 is not None:
    print(f"DBSCAN - eps={best_eps_1}, min_samples={best_min_samples_1}: Silhouette={best_silhouette_dbscan_1:.3f}, DB={db_dbscan_1:.3f}, CH={ch_dbscan_1:.3f}, Noise={noise_ratio_1:.3f}")
print(f"Agglomerative - k={best_k_agg_1}, linkage={best_linkage_1}: Silhouette={best_silhouette_agg_1:.3f}, DB={db_agg_1:.3f}, CH={ch_agg_1:.3f}")

# Выбор лучшего метода для датасета 1
scores_1 = {
    'KMeans': silhouette_kmeans_1,
    'DBSCAN': best_silhouette_dbscan_1 if best_labels_dbscan_1 is not None else -2,
    'Agglomerative': best_silhouette_agg_1
}
best_method_1 = max(scores_1, key=scores_1.get)
print(f"Best method for DS1: {best_method_1} (Silhouette: {scores_1[best_method_1]:.3f})")

print("\n=== DATASET 2 SUMMARY ===")
print(f"KMeans - k={optimal_k_2}: Silhouette={silhouette_kmeans_2:.3f}, DB={db_kmeans_2:.3f}, CH={ch_kmeans_2:.3f}")
if best_labels_dbscan_2 is not None:
    print(f"DBSCAN - eps={best_eps_2}, min_samples={best_min_samples_2}: Silhouette={best_silhouette_dbscan_2:.3f}, DB={db_dbscan_2:.3f}, CH={ch_dbscan_2:.3f}, Noise={noise_ratio_2:.3f}")
print(f"Agglomerative - k={best_k_agg_2}, linkage={best_linkage_2}: Silhouette={best_silhouette_agg_2:.3f}, DB={db_agg_2:.3f}, CH={ch_agg_2:.3f}")

# Выбор лучшего метода для датасета 2
scores_2 = {
    'KMeans': silhouette_kmeans_2,
    'DBSCAN': best_silhouette_dbscan_2 if best_labels_dbscan_2 is not None else -2,
    'Agglomerative': best_silhouette_agg_2
}
best_method_2 = max(scores_2, key=scores_2.get)
print(f"Best method for DS2: {best_method_2} (Silhouette: {scores_2[best_method_2]:.3f})")

print("\n=== DATASET 3 SUMMARY ===")
print(f"KMeans - k={optimal_k_3}: Silhouette={silhouette_kmeans_3:.3f}, DB={db_kmeans_3:.3f}, CH={ch_kmeans_3:.3f}")
if best_labels_dbscan_3 is not None:
    print(f"DBSCAN - eps={best_eps_3}, min_samples={best_min_samples_3}: Silhouette={best_silhouette_dbscan_3:.3f}, DB={db_dbscan_3:.3f}, CH={ch_dbscan_3:.3f}, Noise={noise_ratio_3:.3f}")
print(f"Agglomerative - k={best_k_agg_3}, linkage={best_linkage_3}: Silhouette={best_silhouette_agg_3:.3f}, DB={db_agg_3:.3f}, CH={ch_agg_3:.3f}")

# Выбор лучшего метода для датасета 3
scores_3 = {
    'KMeans': silhouette_kmeans_3,
    'DBSCAN': best_silhouette_dbscan_3 if best_labels_dbscan_3 is not None else -2,
    'Agglomerative': best_silhouette_agg_3
}
best_method_3 = max(scores_3, key=scores_3.get)
print(f"Best method for DS3: {best_method_3} (Silhouette: {scores_3[best_method_3]:.3f})")

## Сохранение артефактов

In [ ]:
import json

# Сохранение сводки метрик
metrics_summary = {
    'dataset_1': {
        'kmeans': {
            'silhouette': float(silhouette_kmeans_1),
            'davies_bouldin': float(db_kmeans_1),
            'calinski_harabasz': float(ch_kmeans_1)
        },
        'dbscan': {
            'silhouette': float(best_silhouette_dbscan_1) if best_labels_dbscan_1 is not None else None,
            'davies_bouldin': float(db_dbscan_1) if best_labels_dbscan_1 is not None else None,
            'calinski_harabasz': float(ch_dbscan_1) if best_labels_dbscan_1 is not None else None,
            'noise_ratio': float(noise_ratio_1) if best_labels_dbscan_1 is not None else None
        },
        'agglomerative': {
            'silhouette': float(best_silhouette_agg_1),
            'davies_bouldin': float(db_agg_1),
            'calinski_harabasz': float(ch_agg_1)
        }
    },
    'dataset_2': {
        'kmeans': {
            'silhouette': float(silhouette_kmeans_2),
            'davies_bouldin': float(db_kmeans_2),
            'calinski_harabasz': float(ch_kmeans_2)
        },
        'dbscan': {
            'silhouette': float(best_silhouette_dbscan_2) if best_labels_dbscan_2 is not None else None,
            'davies_bouldin': float(db_dbscan_2) if best_labels_dbscan_2 is not None else None,
            'calinski_harabasz': float(ch_dbscan_2) if best_labels_dbscan_2 is not None else None,
            'noise_ratio': float(noise_ratio_2) if best_labels_dbscan_2 is not None else None
        },
        'agglomerative': {
            'silhouette': float(best_silhouette_agg_2),
            'davies_bouldin': float(db_agg_2),
            'calinski_harabasz': float(ch_agg_2)
        }
    },
    'dataset_3': {
        'kmeans': {
            'silhouette': float(silhouette_kmeans_3),
            'davies_bouldin': float(db_kmeans_3),
            'calinski_harabasz': float(ch_kmeans_3)
        },
        'dbscan': {
            'silhouette': float(best_silhouette_dbscan_3) if best_labels_dbscan_3 is not None else None,
            'davies_bouldin': float(db_dbscan_3) if best_labels_dbscan_3 is not None else None,
            'calinski_harabasz': float(ch_dbscan_3) if best_labels_dbscan_3 is not None else None,
            'noise_ratio': float(noise_ratio_3) if best_labels_dbscan_3 is not None else None
        },
        'agglomerative': {
            'silhouette': float(best_silhouette_agg_3),
            'davies_bouldin': float(db_agg_3),
            'calinski_harabasz': float(ch_agg_3)
        }
    }
}

with open('artifacts/metrics_summary.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, indent=2, ensure_ascii=False)

print("Metrics summary saved to artifacts/metrics_summary.json")

In [ ]:
# Сохранение лучших конфигураций
best_configs = {
    'dataset_1': {
        'best_method': best_method_1,
        'kmeans_config': {
            'n_clusters': optimal_k_1
        },
        'dbscan_config': {
            'eps': best_eps_1,
            'min_samples': best_min_samples_1
        } if best_labels_dbscan_1 is not None else None,
        'agglomerative_config': {
            'n_clusters': best_k_agg_1,
            'linkage': best_linkage_1
        }
    },
    'dataset_2': {
        'best_method': best_method_2,
        'kmeans_config': {
            'n_clusters': optimal_k_2
        },
        'dbscan_config': {
            'eps': best_eps_2,
            'min_samples': best_min_samples_2
        } if best_labels_dbscan_2 is not None else None,
        'agglomerative_config': {
            'n_clusters': best_k_agg_2,
            'linkage': best_linkage_2
        }
    },
    'dataset_3': {
        'best_method': best_method_3,
        'kmeans_config': {
            'n_clusters': optimal_k_3
        },
        'dbscan_config': {
            'eps': best_eps_3,
            'min_samples': best_min_samples_3
        } if best_labels_dbscan_3 is not None else None,
        'agglomerative_config': {
            'n_clusters': best_k_agg_3,
            'linkage': best_linkage_3
        }
    }
}

with open('artifacts/best_configs.json', 'w', encoding='utf-8') as f:
    json.dump(best_configs, f, indent=2, ensure_ascii=False)

print("Best configurations saved to artifacts/best_configs.json")

In [ ]:
# Сохранение меток для лучшего решения каждого датасета
if best_method_1 == 'KMeans':
    labels_1_final = labels_kmeans_1
elif best_method_1 == 'DBSCAN' and best_labels_dbscan_1 is not None:
    labels_1_final = best_labels_dbscan_1
else:
    labels_1_final = best_labels_agg_1

if best_method_2 == 'KMeans':
    labels_2_final = labels_kmeans_2
elif best_method_2 == 'DBSCAN' and best_labels_dbscan_2 is not None:
    labels_2_final = best_labels_dbscan_2
else:
    labels_2_final = best_labels_agg_2

if best_method_3 == 'KMeans':
    labels_3_final = labels_kmeans_3
elif best_method_3 == 'DBSCAN' and best_labels_dbscan_3 is not None:
    labels_3_final = best_labels_dbscan_3
else:
    labels_3_final = best_labels_agg_3

# Создание DataFrame с метками
labels_df_1 = pd.DataFrame({'sample_id': sample_ids_1, 'cluster_label': labels_1_final})
labels_df_2 = pd.DataFrame({'sample_id': sample_ids_2, 'cluster_label': labels_2_final})
labels_df_3 = pd.DataFrame({'sample_id': sample_ids_3, 'cluster_label': labels_3_final})

# Сохранение меток
labels_df_1.to_csv('artifacts/labels/labels_hw07_ds1.csv', index=False)
labels_df_2.to_csv('artifacts/labels/labels_hw07_ds2.csv', index=False)
labels_df_3.to_csv('artifacts/labels/labels_hw07_ds3.csv', index=False)

print("Cluster labels saved to artifacts/labels/")
print(f"Dataset 1: {labels_df_1.shape[0]} samples, {len(set(labels_1_final))} clusters")
print(f"Dataset 2: {labels_df_2.shape[0]} samples, {len(set(labels_2_final))} clusters")
print(f"Dataset 3: {labels_df_3.shape[0]} samples, {len(set(labels_3_final))} clusters")